# 02 · Key IDs, exposure status and holder snapshots

Inputs: `utxo_life`, `address_stats`, `supply_by_type` (01)

Outputs: `address_key`, `key_exposure`, `holder_snapshot`

| Step | Table | Content |
|---|---|---|
| 1 | `address_key` | address → key_id (P2PK, P2PKH and P2WPKH of the same key merged) |
| 2 | `key_exposure` | date and channel of the first public revelation of each key's public key |
| 3 | `holder_snapshot` | per event, at the close of t−1: balance, latest incoming transfer and group L / T / C of each key |

Exposure is determined by whether the controlling public key has been revealed, not by address prefix:

- P2PK, bare multisig: public key in the output script; exposed at creation
- Taproot (all P2TR): the output key Q is attackable with Shor's algorithm; exposed at creation, separate group T
- P2PKH, P2WPKH, P2SH, P2WSH: any spend from the same key_id reveals the public key or script; coins remaining after a spend are exposed
- Exposed → L; hash-protected and never revealed → C (control group)

key_id namespaces (first byte): `01` public-key hash160 (shared by P2PK, P2PKH and P2WPKH, so reuse across types is linked), `02` P2SH script hash, `03` P2WSH script hash, `04` Taproot Q.

Known limitation: the P2SH-P2WPKH script hash is hash160(0014‖hash160(pk)). BigQuery has no RIPEMD160, so these keys are not linked to P2PKH/P2WPKH addresses of the same key, and exposure is slightly underestimated.

Snapshots keep every key with a positive balance; sample thresholds (0.1 or 0.01 BTC) are applied downstream.

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

# Cost guard: queries estimated below AUTO_RUN_USD run directly; more expensive ones need CONFIRM = True
AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25     # on-demand list price
CONFIRM           = False
FORCE_REBUILD     = False

# Events and snapshots: exposure status and balances are fixed at the close of the day before the event (t-1)
PRE_DAYS = 1
EVENTS = [
    # (label,     event date,    verified)
    ("willow",   "2024-12-09", True),
    ("gidney",   "2025-05-21", True),
    ("lopp",     "2025-07-15", True),
    ("bip360",   "2026-02-11", True),    # BIP-360 (P2MR) merged into bitcoin/bips; dated like BIP-361
    # first BIP-360 draft (PR #1670, numbered 2024-12-18) falls in the Willow window; not a separate event
    ("bip360_draft", "2024-09-27", True),
    ("google",   "2026-03-31", True),
    ("bip361",   "2026-04-14", True),
]
LATEST_SNAP = "2026-09-22"   # current-stock snapshot (freeze date)
MIN_BTC     = 0.1            # threshold used in the summaries only; does not affect the tables

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, time, datetime as dt
import pandas as pd

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS = f"{PROJECT_ID}.{DATASET}"

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

for t in ["utxo_life", "address_stats", "supply_by_type"]:
    client.get_table(f"{DS}.{t}")        # raises if a table is missing
print("Tables from 01 found. Spent so far at list price: $%.2f / $%.0f" % (LEDGER["usd_total"], MAX_USD_TOTAL))

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 1)}

def build(name, sql, force=False):
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}; set CONFIRM=True and re-run")
        return

    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id} (runs on the server; finishes even if Colab disconnects)")

    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 1),
                           "usd": round(usd(billed), 2), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.0f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))

def q(sql):
    return client.query(sql).to_dataframe()

## Step 1 · `address_key`

A JavaScript UDF decodes each address to a key_id (base58 and bech32, offline; tested on the BIP-173 test vectors) and is applied once to the 1.55 billion addresses in `address_stats`.

In [ ]:
UDF_JS = r"""
function keyid(addr){
  if (addr === null || addr === undefined) return null;
  var hex = function(a){ var h=''; for (var i=0;i<a.length;i++){ h += (a[i]<16?'0':'') + a[i].toString(16);} return h; };
  var s = String(addr);
  if (s.charAt(0) === '1' || s.charAt(0) === '3') {
    var A = '123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz';
    var b = [];
    for (var i=0;i<s.length;i++){
      var carry = A.indexOf(s.charAt(i)); if (carry < 0) return null;
      for (var j=0;j<b.length;j++){ carry += b[j]*58; b[j] = carry & 255; carry >>= 8; }
      while (carry > 0){ b.push(carry & 255); carry >>= 8; }
    }
    for (var k=0;k<s.length && s.charAt(k)==='1';k++) b.push(0);
    b.reverse();
    if (b.length !== 25) return null;
    if (b[0] === 0) return '01' + hex(b.slice(1,21));
    if (b[0] === 5) return '02' + hex(b.slice(1,21));
    return null;
  }
  var t = s.toLowerCase();
  if (t.indexOf('bc1') !== 0) return null;
  var C = 'qpzry9x8gf2tvdw0s3jn54khce6mua7l';
  var d = [];
  for (var i2=3;i2<t.length;i2++){ var v = C.indexOf(t.charAt(i2)); if (v<0) return null; d.push(v); }
  if (d.length < 8) return null;
  var ver = d[0], p = d.slice(1, d.length-6), out = [], acc = 0, bits = 0;
  for (var m=0;m<p.length;m++){ acc = ((acc << 5) | p[m]) & 0xfff; bits += 5; while (bits >= 8){ bits -= 8; out.push((acc >> bits) & 255); } }
  if (ver === 0 && out.length === 20) return '01' + hex(out);
  if (ver === 0 && out.length === 32) return '03' + hex(out);
  if (ver === 1 && out.length === 32) return '04' + hex(out);
  return null;
}

return keyid(addr);
"""

client.query(f"""
CREATE OR REPLACE FUNCTION `{DS}.key_id_hex`(addr STRING)
RETURNS STRING
LANGUAGE js AS r\"\"\"{UDF_JS}\"\"\"
""").result()

# self-test: four known addresses
chk = q(f"""
SELECT a, `{DS}.key_id_hex`(a) AS key_hex FROM UNNEST([
  '1A1zP1eP5QGefi2DMPTfTL5SLmv7DivfNa',
  'bc1qw508d6qejxtdg4y5r3zarvary0c5xw7kv8f3t4',
  'bc1qrp33g0q5c5txsp9arysrx4k6zdkfs4nce4xj0gdcccefvpysxf3qccfmv3',
  'bc1p0xlxvlhemja6c4dqv22uapctqupfhlxm9h8z3k2e72q4k9hcz7vqzk5jj0']) a
""")
expected = ["0162e907b15cbf27d5425399ebf6f0fb50ebb88f18",
            "01751e76e8199196d454941c45d1b3a323f1433bd6",
            "031863143c14c5166804bd19203356da136c985678cd4d27a1b8c6329604903262",
            "0479be667ef9dcbbac55a06295ce870b07029bfcdb2dce28d959f2815b16f81798"]
assert list(chk.key_hex) == expected, chk
print("UDF self-test passed")

In [ ]:
SQL_ADDRESS_KEY = f'''
CREATE OR REPLACE TABLE `{DS}.address_key`
CLUSTER BY address
AS
WITH x AS (
  SELECT address, `{DS}.key_id_hex`(address) AS h
  FROM `{DS}.address_stats`
)
SELECT
  address,
  FROM_HEX(h) AS key_id,
  CASE LEFT(h, 2) WHEN '01' THEN 'k' WHEN '02' THEN 's'
                  WHEN '03' THEN 'w' WHEN '04' THEN 't' END AS key_kind
FROM x
'''
build("address_key", SQL_ADDRESS_KEY)

## Step 2 · `key_exposure`

First revelation date of each key's public key, the earliest of:

1. `spend`: first spend from any address of the same key_id (address reuse, reuse across types, script reuse)
2. `p2pk_output` / `multisig_output`: public key in the output script
3. `taproot_output`: Taproot output, exposed at creation

Keys never exposed are absent (NULL after the LEFT JOIN in step 3 → group C).

In [ ]:
SQL_KEY_EXPOSURE = f'''
CREATE OR REPLACE TABLE `{DS}.key_exposure`
AS
WITH ev AS (
  SELECT k.key_id, a.first_spend_date AS d, 'spend' AS src
  FROM `{DS}.address_stats` a
  JOIN `{DS}.address_key`  k USING (address)
  WHERE a.first_spend_date IS NOT NULL AND k.key_id IS NOT NULL

  UNION ALL
  SELECT FROM_HEX(`{DS}.key_id_hex`(address)), created_date, 'p2pk_output'
  FROM `{DS}.utxo_life`
  WHERE type = 'pubkey' AND address IS NOT NULL

  UNION ALL
  SELECT FROM_HEX(`{DS}.key_id_hex`(m)), created_date, 'multisig_output'
  FROM `{DS}.utxo_life`, UNNEST(multisig_addresses) AS m
  WHERE type = 'multisig'

  UNION ALL
  SELECT k.key_id, a.first_seen_date, 'taproot_output'
  FROM `{DS}.address_stats` a
  JOIN `{DS}.address_key`  k USING (address)
  WHERE k.key_kind = 't'
)
SELECT
  key_id,
  MIN(d) AS first_exposed_date,
  ARRAY_AGG(src ORDER BY d, src LIMIT 1)[OFFSET(0)] AS first_exposed_src
FROM ev
WHERE key_id IS NOT NULL
GROUP BY key_id
'''
build("key_exposure", SQL_KEY_EXPOSURE)

## Step 3 · `holder_snapshot`

Snapshot date = t−1. UTXOs alive at the close of the snapshot date (`created_date ≤ snapshot date`, unspent or spent later) are aggregated by key_id. Groups: Taproot → T; exposed on or before the snapshot date → L; otherwise → C.

`last_in_date` is the latest incoming transfer before the snapshot date, used for the risk set (incoming transfer within 12 months). Only receipts in the last 12 months and UTXOs still held are read, so the value is exact within 12 months and otherwise only means "more than 12 months ago".

Six standard types only (pubkey, pubkeyhash, witness_v0_keyhash, scripthash, witness_v0_scripthash, witness_v1_taproot); nonstandard, bare multisig and witness_unknown outputs hold less than 3,000 BTC in total.

In [ ]:
snaps = [(lab, ev, (pd.Timestamp(ev) - pd.Timedelta(days=PRE_DAYS)).strftime("%Y-%m-%d"), ok)
         for lab, ev, ok in EVENTS] + [("latest", LATEST_SNAP, LATEST_SNAP, True)]
display(pd.DataFrame(snaps, columns=["event", "event_date", "snap_date", "verified"]))

SNAPS_SQL = ",\n    ".join(
    f"STRUCT('{lab}' AS event, DATE '{ev}' AS event_date, DATE '{sd}' AS snap_date)"
    for lab, ev, sd, _ in snaps)

SQL_SNAPSHOT = f'''
CREATE OR REPLACE TABLE `{DS}.holder_snapshot`
CLUSTER BY event, grp
AS
WITH snaps AS (
  SELECT * FROM UNNEST([
    {SNAPS_SQL}
  ])
),
held AS (
  SELECT s.event, s.event_date, s.snap_date, u.address, u.type, u.sats, u.created_date,
         (u.spent_date IS NULL OR u.spent_date > s.snap_date) AS alive
  FROM `{DS}.utxo_life` u
  CROSS JOIN snaps s
  WHERE u.type IN ('pubkey','pubkeyhash','witness_v0_keyhash','scripthash',
                   'witness_v0_scripthash','witness_v1_taproot')
    AND u.address IS NOT NULL
    AND u.created_date <= s.snap_date
    AND ( u.spent_date IS NULL OR u.spent_date > s.snap_date
          OR u.created_date > DATE_SUB(s.snap_date, INTERVAL 365 DAY) )
),
agg AS (
  SELECT
    a.event, a.event_date, a.snap_date, k.key_id,
    ANY_VALUE(k.key_kind)                          AS key_kind,
    SUM(IF(a.alive, a.sats, 0))                    AS sats,
    COUNTIF(a.alive)                               AS n_utxo,
    COUNT(DISTINCT IF(a.alive, a.address, NULL))   AS n_addr,
    MIN(IF(a.alive, a.created_date, NULL))         AS oldest_utxo_date,
    MAX(IF(a.alive, a.created_date, NULL))         AS newest_utxo_date,
    MAX(a.created_date)                            AS last_in_date,
    ARRAY_AGG(DISTINCT IF(a.alive, a.type, NULL) IGNORE NULLS)         AS types
  FROM held a
  JOIN `{DS}.address_key` k USING (address)
  WHERE k.key_id IS NOT NULL
  GROUP BY a.event, a.event_date, a.snap_date, k.key_id
  HAVING SUM(IF(a.alive, a.sats, 0)) > 0
)
SELECT
  g.*,
  e.first_exposed_date,
  e.first_exposed_src,
  CASE WHEN g.key_kind = 't'                       THEN 'T'
       WHEN e.first_exposed_date <= g.snap_date    THEN 'L'
       ELSE 'C' END AS grp
FROM agg g
LEFT JOIN `{DS}.key_exposure` e USING (key_id)
'''
build("holder_snapshot", SQL_SNAPSHOT)

## Checks

1. key_id coverage: addresses without a `key_id` are limited to a few unusual formats
2. Stock: total BTC in the `latest` snapshot ≈ the six standard types in `supply_by_type` (one day apart; a gap of a few hundred BTC is expected)
3. Group L contains all P2PK (about 1.71 million BTC); group T ≈ Taproot stock (about 0.218 million BTC)

In [ ]:
print("1) key_id coverage")
display(q(f'''
SELECT key_kind, COUNT(*) AS n_addr
FROM `{DS}.address_key` GROUP BY key_kind ORDER BY n_addr DESC
'''))

print("2) Stock reconciliation (latest snapshot vs supply_by_type)")
std = "('pubkey','pubkeyhash','witness_v0_keyhash','scripthash','witness_v0_scripthash','witness_v1_taproot')"
ref = q(f"SELECT SUM(btc_unspent) AS btc FROM `{DS}.supply_by_type` WHERE type IN {std}").btc[0]
got = q(f"SELECT SUM(sats)/1e8 AS btc FROM `{DS}.holder_snapshot` WHERE event='latest'").btc[0]
print(f"   supply_by_type, six types {ref:,.0f} BTC | latest snapshot {got:,.0f} BTC | difference {got-ref:+,.0f}")

print("3) Snapshots x groups")
summary = q(f'''
SELECT event, snap_date, grp,
  COUNT(*)                                        AS n_keys,
  ROUND(SUM(sats)/1e8)                            AS btc,
  COUNTIF(sats >= {MIN_BTC}*1e8)                  AS n_keys_ge_min,
  ROUND(SUM(IF(sats >= {MIN_BTC}*1e8, sats, 0))/1e8) AS btc_ge_min,
  COUNTIF(sats >= {MIN_BTC}*1e8
          AND last_in_date > DATE_SUB(snap_date, INTERVAL 365 DAY)) AS n_keys_riskset
FROM `{DS}.holder_snapshot`
GROUP BY event, snap_date, grp
ORDER BY snap_date, grp
''')
display(summary)

print("4) latest snapshot, group L by exposure channel")
display(q(f'''
SELECT first_exposed_src, COUNT(*) AS n_keys, ROUND(SUM(sats)/1e8) AS btc
FROM `{DS}.holder_snapshot`
WHERE event='latest' AND grp='L'
GROUP BY 1 ORDER BY btc DESC
'''))

In [ ]:
for rec in JOBS[-5:]:
    job = client.get_job(rec["job_id"], location=rec.get("location", LOCATION))
    print(rec["table"], job.state, "errors:", job.errors)
    if job.state != "DONE":
        job.result(); print("  done")
for t in ["address_key", "key_exposure", "holder_snapshot"]:
    n = table_rows(t)
    print(f"{t:16s}", f"{n:,} rows" if n else "not built yet")

In [ ]:
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
          "snapshots": snaps, "pre_days": PRE_DAYS, "ledger": LEDGER,
          "tables": [table_info(t) for t in ["address_key", "key_exposure", "holder_snapshot"]]}
try:
    report["summary"] = summary.astype(str).to_dict(orient="records")
except NameError:
    pass
with open("build_report_02.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
print(json.dumps(report["tables"], ensure_ascii=False, indent=2))
try:
    from google.colab import files
    files.download("build_report_02.json")
except Exception:
    print("Report saved to build_report_02.json")